In [1]:

import pandas as pd

email_batch_1 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch1_10k.csv')
email_batch_2 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch2_10k.csv')
email_batch_3 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch3_10k.csv')
email_batch_4 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch4_10k.csv')
email_batch_5 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch5_10k.csv')
email_batch_6 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch6_10k.csv')
email_batch_7 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch7_10k.csv')
email_batch_8 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch8_10k.csv')
email_batch_9 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch9_10k.csv')
email_batch_10 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/refined_priority_batch10_10k.csv')
email_batch_11 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/cleaned_priority_synthetic_balanced_full_dataset.csv')
email_batch_12 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/generated_email_dataset_10k_1136_2142025.csv')
email_batch_13 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/it_support_email_priority.csv')
email_batch_14 = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Model Building/A_Dataset/Priority/emails_dataset_20k_py.csv')

email_data = pd.concat([email_batch_1, email_batch_2, email_batch_3, email_batch_4, email_batch_5, email_batch_6, email_batch_7, email_batch_8, email_batch_9, email_batch_10, email_batch_11, email_batch_12, email_batch_13, email_batch_14], ignore_index=True)
email_data.columns

Index(['Email ID', 'From', 'To', 'Date & Time', 'Subject', 'Message',
       'Priority'],
      dtype='object')

In [2]:
email_data['Priority'].value_counts()

Priority
Medium    51800
Low       44080
High      43912
Name: count, dtype: int64

In [3]:
# Check for duplicates based on Subject + Message
duplicate_subject_msg = email_data[email_data.duplicated(subset=['Subject', 'Message'])]
print(f"📧 Duplicates based on Subject + Message: {duplicate_subject_msg.shape[0]}")

📧 Duplicates based on Subject + Message: 9638


In [4]:
email_data = email_data.drop_duplicates(subset=['Subject', 'Message'], keep='first')

print(f"✅ New dataset shape after removing duplicates: {email_data.shape}")

✅ New dataset shape after removing duplicates: (130155, 7)


In [6]:
import re
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # Only tokenizer + lemmatizer

def advanced_cleaning_pipeline(email_text):
    if not isinstance(email_text, str):
        return ""

    # 1. Remove forwarded/original headers (multi-line removal)
    email_text = re.sub(r"(?is)-----.*?Subject:", "", email_text)

    # 2. Normalize all newlines, tabs, excessive spaces
    email_text = re.sub(r"[\r\n\t]+", " ", email_text)
    email_text = re.sub(r"\s+", " ", email_text).strip()

    # 3. Fix common email encoding artifacts
    email_text = re.sub(r"=20", " ", email_text)  # Quoted printable space
    email_text = re.sub(r"=09", " ", email_text)  # Quoted printable tab
    email_text = re.sub(r"=3D", "=", email_text)  # Quoted printable equals
    email_text = re.sub(r"=\s?", "", email_text)  # Quoted printable soft line break

    # 4. Remove URLs, Emails, Attachments (no placeholders - full removal)
    email_text = re.sub(r"http[s]?://\S+", "", email_text)
    email_text = re.sub(r"\b\S+@\S+\b", "", email_text)
    email_text = re.sub(r"<< File:.*?>>", "", email_text)

    # 5. Remove common signatures/closings
    email_text = re.sub(r"(?i)(thanks|regards|sincerely|best),?", "", email_text)

    # 6. Optional: Remove phone numbers
    email_text = re.sub(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", "", email_text)

    # 7. Final cleanup - extra spaces from removals
    email_text = re.sub(r"\s+", " ", email_text).strip()

    # 8. Run spaCy NLP (tokenization, lemmatization, stopword removal)
    doc = nlp(email_text)

    # 9. Keep only lemmatized tokens (no stopwords, no punctuation, no numbers)
    cleaned_tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.like_num
    ]

    cleaned_text = " ".join(cleaned_tokens)

    return cleaned_text

email_data['Cleaned_Message'] = email_data['Message'].apply(advanced_cleaning_pipeline)

In [5]:
import re
def remove_duplicate_sentences(text):
    if not isinstance(text, str):
        return ""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen = set()
    cleaned = []
    for s in sentences:
        norm = re.sub(r'[.!?]', '', s.strip().lower())
        if norm not in seen:
            seen.add(norm)
            cleaned.append(s.strip())
    return ' '.join(cleaned)

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_duplicate_sentences)

In [12]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Optional: Download once
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_subject_line(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'^(FW:|RE:|Fwd:|Re:|\[.*?\])', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def tokenize_and_filter(text):
    # Tokenize and remove stopwords (optional depending on task)
    tokens = word_tokenize(text)
    filtered = [token for token in tokens if token not in stop_words]
    return " ".join(filtered)

# Example usage:
email_data['Cleaned_Subject'] = email_data['Subject'].apply(clean_subject_line)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
def remove_emojis(text):
    # Regular expression pattern to match emojis
    emoji_pattern = re.compile("[\U0001F600-\U0001F64F|\U0001F300-\U0001F5FF|\U0001F680-\U0001F6FF|\U0001F700-\U0001F77F|\U0001F780-\U0001F7FF|\U0001F800-\U0001F8FF|\U0001F900-\U0001F9FF|\U0001FA00-\U0001FA6F|\U0001FA70-\U0001FAFF|\U00002702-\U000027B0|\U000024C2-\U0001F251]")
    return re.sub(emoji_pattern, '', text)

# Apply the function to remove emojis from the Cleaned_Subject column
email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(remove_emojis)
email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_emojis)

C:\Users\User\AppData\Local\Temp\ipykernel_10680\3803423692.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(remove_emojis)
C:\Users\User\AppData\Local\Temp\ipykernel_10680\3803423692.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_emojis)


In [13]:
def clean_subject_line(text):
    if not isinstance(text, str):
        return ""

    # Remove email reply/forward prefixes
    text = re.sub(r'^(FW:|RE:|Fwd:|Re:|\[.*?\])', '', text, flags=re.IGNORECASE)

    # Remove emojis
    emoji_pattern = re.compile(
        "[" 
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    # Normalize spacing
    return re.sub(r'\s+', ' ', text).strip().lower()
email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(clean_subject_line)

In [14]:
def clean_message_text(text):
    if not isinstance(text, str):
        return ""

    # Remove email addresses
    text = re.sub(r'\b\S+@\S+\b', '', text)

    # Remove URLs
    text = re.sub(r'\b(?:http|https|www)\S*', '', text)

    # Remove emojis
    emoji_pattern = re.compile(
        "[" 
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    # Normalize whitespace
    return re.sub(r'\s+', ' ', text).strip().lower()

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(clean_message_text)

In [15]:
import re

def ml_clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove emails and URLs
    text = re.sub(r'\b\S+@\S+\b', '', text)                      # email addresses
    text = re.sub(r'\b(?:http|https|www)\S*\b', '', text)        # URLs

    # 2. Remove emojis
    emoji_pattern = re.compile(
        "[" 
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)

    # 3. Remove unwanted symbols (keep .,!?%/- for meaning)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?%/:-]', '', text)

    # 4. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Lowercase
    return text.lower()

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(clean_message_text)

In [16]:
import re

def remove_duplicate_sentences(text):
    if not isinstance(text, str):
        return ""
    sentences = re.split(r'(?<=[.!?]) +', text)
    seen = set()
    unique_sentences = []
    for s in sentences:
        if s not in seen:
            unique_sentences.append(s)
            seen.add(s)
    return ' '.join(unique_sentences)

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_duplicate_sentences)

In [17]:
import re

def remove_duplicate_sentences(text):
    if not isinstance(text, str):
        return ""
    
    # Split into sentences using punctuation
    sentences = re.split(r'(?<=[.!?]) +', text)
    seen = set()
    unique_sentences = []
    
    for s in sentences:
        # Normalize sentence (remove trailing punctuation + extra space)
        normalized = s.strip().lower()
        normalized = re.sub(r'[.!?]+$', '', normalized)  # remove end punctuation for matching
        
        if normalized not in seen:
            seen.add(normalized)
            unique_sentences.append(s.strip())  # keep original sentence for output

    return ' '.join(unique_sentences)

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_duplicate_sentences)

In [18]:
def remove_duplicate_sentences(text):
    if not isinstance(text, str):
        return ""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    seen = set()
    cleaned = []
    for s in sentences:
        norm = re.sub(r'[.!?]+$', '', s.strip().lower())
        if norm not in seen:
            seen.add(norm)
            cleaned.append(s.strip())
    return ' '.join(cleaned)

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_duplicate_sentences)

In [19]:
def remove_duplicate_sentences_strict(text):
    if not isinstance(text, str):
        return ""
    sentences = re.split(r'(?<=[.!?]) +', text)
    seen = set()
    cleaned = []
    for s in sentences:
        normalized = re.sub(r'[.!?]', '', s.strip().lower())
        if normalized not in seen:
            seen.add(normalized)
            cleaned.append(s.strip())
    return ' '.join(cleaned)

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_duplicate_sentences_strict)

In [ ]:
email_data

In [20]:
# Check for duplicates based on Subject + Message
duplicate_subject_msg = email_data[email_data.duplicated(subset=['Cleaned_Subject', 'Cleaned_Message'])]
print(f"📧 Duplicates based on Subject + Message: {duplicate_subject_msg.shape[0]}")

📧 Duplicates based on Subject + Message: 8912


In [21]:
email_data = email_data.drop_duplicates(subset=['Cleaned_Subject', 'Cleaned_Message'], keep='first')

print(f"✅ New dataset shape after removing duplicates: {email_data.shape}")

✅ New dataset shape after removing duplicates: (121243, 9)


In [ ]:
email_data

In [23]:
email_data.to_csv('test_cleaned_priority_email_dataset_machine_learning_model.csv', index=False)